In [1]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from data_model.manage_excel_support_file import *
from data_model.MergerTools import *
import pandas as pd
import os

client = DatalakeClient()
mergeTools = MergerTools()

In [ ]:
# ===================================================================
# Step 1: Setup and Data Loading
# ===================================================================
print("=" * 80)
print("STEP 1: LOADING DATA")
print("=" * 80)

print("\nLoading df_merge...")
df_merge = client.download_file('cleaned/merged/combination/subMERGE_4-3_elecsys_FBP.csv')
print(f"Loaded {len(df_merge)} rows, {len(df_merge.columns)} columns")

print("\nLoading df_demog (DXSUM)...")
search = client.query_files(query={'custom.level': 'cleaned_01', 'custom.file_code': 'PTDEMOG'})
zip_files = client.download_file(search['object_name'], extract_zip=True)
df_demog = zip_files[list(zip_files.keys())[0]].copy(deep=True)
print(f"Loaded {len(df_demog)} rows, {len(df_demog.columns)} columns")

# Define column groups
cols_check = {
    'DX': ['DX/CN', 'DX/Dementia', 'DX/MCI'],
    'ETHNICITY': ['ETHNICITY/latino', 'ETHNICITY/not_latino'],
    'MARRY': ['MARRY/divorced', 'MARRY/married', 'MARRY/single', 'MARRY/widowed'],
    'RACE': ['RACE/Asian', 'RACE/Black', 'RACE/Mixed', 'RACE/Native_american', 'RACE/White']
}

print("\nColumn groups defined:")
for cat, cols in cols_check.items():
    print(f"  {cat}: {len(cols)} columns")


In [ ]:

# ===================================================================
# Step 2: Standardize Data Types
# ===================================================================
print("\n" + "=" * 80)
print("STEP 2: STANDARDIZING DATA TYPES")
print("=" * 80)

all_dummy_cols = []
for cols_list in cols_check.values():
    all_dummy_cols.extend(cols_list)

print(f"\nConverting {len(all_dummy_cols)} dummy columns to float...")

for col in all_dummy_cols:
    if col in df_merge.columns:
        df_merge[col] = pd.to_numeric(
            df_merge[col].replace({'True': 1, 'False': 0, 'true': 1, 'false': 0}),
            errors='coerce'
        )
    else:
        print(f"  Warning: Column '{col}' not found in df_merge")

print("Data type conversion complete!")


In [ ]:
# ===================================================================
# Step 3: Verify RACE Mapping
# ===================================================================
print("\n" + "=" * 80)
print("STEP 3: VERIFYING RACE MAPPING")
print("=" * 80)

race_cols = cols_check['RACE']
valid_race_mask = df_merge[race_cols].sum(axis=1) > 0
df_valid_race = df_merge[valid_race_mask].copy()
print(f"\nFound {len(df_valid_race)} rows with valid RACE values")

comparison = df_valid_race[['RID'] + race_cols].merge(
    df_demog[['RID', 'RACE']], on='RID', how='inner'
)
print(f"Matched {len(comparison)} rows with df_demog")

race_map = {1.0: 'Native_american', 2.0: 'Asian', 4.0: 'Black', 5.0: 'White', 0.0: 'Mixed'}

mismatches = []
for idx, row in comparison.iterrows():
    demog_race = row['RACE']
    if pd.isna(demog_race) or demog_race == 3:
        continue

    expected_col = f"RACE/{race_map.get(int(demog_race))}"
    if expected_col in race_cols and row[expected_col] != 1.0:
        mismatches.append(row['RID'])

print(f"Found {len(mismatches)} mismatches in RACE mapping validation")
if len(mismatches) == 0:
    print("✓ All RACE mappings are correct!")


In [ ]:
# ===================================================================
# Step 4: Initial Problem Analysis
# ===================================================================
print("\n" + "=" * 80)
print("STEP 4: INITIAL PROBLEM ANALYSIS")
print("=" * 80)

for category, cols in cols_check.items():
    # Righe con tutte le colonne NaN
    all_nan_mask = df_merge[cols].isna().all(axis=1)
    num_all_nan = all_nan_mask.sum()
    rids_all_nan = df_merge[all_nan_mask]['RID'].nunique()
    
    # Righe con tutti zeri (escludendo le righe già tutte NaN)
    df_check = df_merge[~all_nan_mask].copy()
    all_zero_mask = (df_check[cols].fillna(0) == 0).all(axis=1)
    num_all_zeros = all_zero_mask.sum()
    rids_all_zeros = df_check.loc[all_zero_mask, 'RID'].nunique()

    print(f"\n{category}:")
    print(f"  - Rows with all NaN: {num_all_nan} ({rids_all_nan} unique RIDs)")
    print(f"  - Rows with all zeros: {num_all_zeros} ({rids_all_zeros} unique RIDs)")

    if rids_all_zeros > 0:
        rids_list = df_check.loc[all_zero_mask, 'RID'].unique()
        in_demog = df_demog[df_demog['RID'].isin(rids_list)]['RID'].nunique()
        print(f"  - RIDs with all zeros present in df_demog: {in_demog}/{rids_all_zeros} ({in_demog/rids_all_zeros*100:.1f}%)")



In [ ]:
# ===================================================================
# Step 5: Fix Simple Categories (ETHNICITY, MARRY, DX)
# ===================================================================
print("\n" + "=" * 80)
print("STEP 5: FIXING SIMPLE CATEGORIES")
print("=" * 80)

for category in ['ETHNICITY', 'MARRY', 'DX']:
    cols = cols_check[category]
    all_zero_mask = (df_merge[cols].fillna(0) == 0).all(axis=1)
    df_merge.loc[all_zero_mask, cols] = np.nan
    print(f"\n{category}: Set {all_zero_mask.sum()} rows to NaN")


In [ ]:

# ===================================================================
# Step 6: Fix RACE Category---> RICALCOLO COLONNE RACE DUMMIES DA df_demog (pulito)
# ===================================================================

print(f"\n{'=' * 80}")
print("RICALCOLO COLONNE RACE DUMMIES DA df_demog")
print(f"{'=' * 80}")

# 1. Backup delle colonne RACE originali (per confronto e fallback)
race_cols = cols_check['RACE']
backup_cols = {}
for col in race_cols:
    backup_cols[col] = df_merge[col].copy()
    
print(f"✓ Backup delle colonne originali creato")

# 2. Filtra df_demog solo per RID presenti in df_merge
rids_in_merge = df_merge['RID'].unique()
df_demog_filtered = df_demog[df_demog['RID'].isin(rids_in_merge)].copy()

print(f"\n✓ df_demog filtrato per RID presenti in df_merge:")
print(f"  - RID unici in df_merge: {len(rids_in_merge)}")
print(f"  - RID trovati in df_demog: {df_demog_filtered['RID'].nunique()}")
print(f"  - RID NON trovati in df_demog: {len(rids_in_merge) - df_demog_filtered['RID'].nunique()}")

# 3. Crea df_race_clean (un valore RACE per RID)
print(f"\n{'─' * 80}")
print("Creazione df_race_clean:")
print(f"{'─' * 80}")

def get_race_cleaned(series):
    """
    Se RACE costante → usa quello
    Se RACE variabile → 0 (Mixed)
    Se RACE tutto NaN → NaN
    """
    non_nan = series.dropna()
    if len(non_nan) == 0:
        return np.nan
    unique_vals = non_nan.unique()
    if len(unique_vals) == 1:
        return unique_vals[0]
    else:
        return 0  # Mixed

df_race_clean = df_demog_filtered.groupby('RID')['RACE'].agg(get_race_cleaned).reset_index()
df_race_clean.columns = ['RID', 'RACE_cleaned']

print(f"✓ df_race_clean creato con {len(df_race_clean)} RID")
print(f"\nDistribuzione RACE_cleaned:")
print(df_race_clean['RACE_cleaned'].value_counts(dropna=False))

# 4. Mapping da valore numerico a nome colonna dummy
race_map = {
    1: 'Native_american', 
    2: 'Asian', 
    3: None,  # Hawaiian/Pacific Islander - non ha colonna
    4: 'Black', 
    5: 'White', 
    0: 'Mixed'
}

# 5. Rigenera le colonne RACE dummies
print(f"\n{'─' * 80}")
print("Rigenerazione colonne RACE dummies:")
print(f"{'─' * 80}")

# Reset tutte le colonne RACE a 0.0 (preparazione)
for col in race_cols:
    df_merge[col] = 0.0

# Contatori
stats = {
    'rigenerati_da_demog': 0,
    'impostati_nan': 0,
    'hawaiian_to_nan': 0,
    'mantenuti_originali': 0
}

# RID presenti in df_race_clean
rids_with_race_clean = set(df_race_clean['RID'].values)

# Per ogni RID in df_merge
for rid in df_merge['RID'].unique():
    mask = df_merge['RID'] == rid
    
    # Caso A: RID presente in df_race_clean
    if rid in rids_with_race_clean:
        race_value = df_race_clean[df_race_clean['RID'] == rid]['RACE_cleaned'].values[0]
        
        if pd.isna(race_value):
            # RACE sconosciuto in df_demog → NaN
            df_merge.loc[mask, race_cols] = np.nan
            stats['impostati_nan'] += mask.sum()
            
        elif race_value == 3:
            # Hawaiian/Pacific Islander → NaN (no colonna)
            df_merge.loc[mask, race_cols] = np.nan
            stats['hawaiian_to_nan'] += mask.sum()
            
        elif int(race_value) in race_map and race_map[int(race_value)] is not None:
            # Valore valido → imposta colonna corretta
            target_col = f"RACE/{race_map[int(race_value)]}"
            df_merge.loc[mask, race_cols] = 0.0
            df_merge.loc[mask, target_col] = 1.0
            stats['rigenerati_da_demog'] += mask.sum()
            
        else:
            # Valore sconosciuto → NaN
            df_merge.loc[mask, race_cols] = np.nan
            stats['impostati_nan'] += mask.sum()
    
    # Caso B: RID NON presente in df_demog → mantieni valori originali
    else:
        for col in race_cols:
            df_merge.loc[mask, col] = backup_cols[col][mask].values
        stats['mantenuti_originali'] += mask.sum()

print(f"\n✓ Rigenerazione completata!")
print(f"\nStatistiche (per righe):")
print(f"  - Righe rigenerate da df_demog: {stats['rigenerati_da_demog']}")
print(f"  - Righe impostate a NaN (no info): {stats['impostati_nan']}")
print(f"  - Righe Hawaiian/PI (→ NaN): {stats['hawaiian_to_nan']}")
print(f"  - Righe con valori originali mantenuti: {stats['mantenuti_originali']}")

# 6. Verifica consistenza: RACE deve essere uguale per tutte le righe dello stesso RID
print(f"\n{'─' * 80}")
print("VERIFICA CONSISTENZA PER RID:")
print(f"{'─' * 80}")

inconsistent_rids = []
for rid in df_merge['RID'].unique():
    rid_data = df_merge[df_merge['RID'] == rid][race_cols]
    
    # Verifica che tutte le righe abbiano gli stessi valori RACE
    if len(rid_data) > 1:
        first_row = rid_data.iloc[0]
        for idx in range(1, len(rid_data)):
            if not rid_data.iloc[idx].equals(first_row):
                inconsistent_rids.append(rid)
                break

if len(inconsistent_rids) == 0:
    print("✓ PERFETTO: Tutti i RID hanno RACE consistente tra le loro righe!")
else:
    print(f"⚠️ ATTENZIONE: {len(inconsistent_rids)} RID hanno RACE inconsistente tra le righe")
    print(f"Primi 5 RID problematici: {inconsistent_rids[:5]}")

# 7. Validazione: verifica che non ci siano più all-zeros
print(f"\n{'─' * 80}")
print("VALIDAZIONE ALL-ZEROS:")
print(f"{'─' * 80}")

def check_all_zeros_race(df, race_cols):
    """Conta righe con tutti zeri nelle colonne RACE (escludendo all-NaN)"""
    all_nan_mask = df[race_cols].isna().all(axis=1)
    df_check = df[~all_nan_mask].copy()
    
    all_zero_mask = (df_check[race_cols].fillna(0) == 0).all(axis=1)
    return all_zero_mask.sum(), df_check.loc[all_zero_mask, 'RID'].nunique()

# Prima (usando backup)
df_backup = df_merge.copy()
for col in race_cols:
    df_backup[col] = backup_cols[col]

num_zeros_before, rids_before = check_all_zeros_race(df_backup, race_cols)
print(f"PRIMA della rigenerazione:")
print(f"  - Righe con tutti zeri: {num_zeros_before} ({rids_before} RID unici)")

# Dopo
num_zeros_after, rids_after = check_all_zeros_race(df_merge, race_cols)
print(f"\nDOPO la rigenerazione:")
print(f"  - Righe con tutti zeri: {num_zeros_after} ({rids_after} RID unici)")

if num_zeros_after == 0:
    print(f"\n✓ PERFETTO: Nessuna riga con tutti zeri!")
elif num_zeros_after < num_zeros_before:
    improvement = num_zeros_before - num_zeros_after
    print(f"\n✓ MIGLIORAMENTO: {improvement} righe corrette ({improvement/num_zeros_before*100:.1f}%)!")
else:
    print(f"\n⚠️ Ancora {num_zeros_after} righe con tutti zeri")

# 8. Verifica consistenza valori: ogni riga deve avere esattamente 1 o tutti NaN
print(f"\n{'─' * 80}")
print("VERIFICA CONSISTENZA VALORI:")
print(f"{'─' * 80}")

sums = df_merge[race_cols].sum(axis=1)
all_nan_mask = df_merge[race_cols].isna().all(axis=1)

consistent = ((sums == 1.0) | all_nan_mask).sum()
inconsistent = (~((sums == 1.0) | all_nan_mask)).sum()

print(f"Righe con esattamente 1 RACE o tutto NaN: {consistent} ({consistent/len(df_merge)*100:.1f}%)")
print(f"Righe inconsistenti: {inconsistent}")

if inconsistent == 0:
    print("\n✓ PERFETTO: Tutte le righe sono consistenti!")
else:
    print(f"\n⚠️ ATTENZIONE: {inconsistent} righe inconsistenti")
    inconsistent_mask = ~((sums == 1.0) | all_nan_mask)
    print("\nPrimi 5 esempi:")
    print(df_merge[inconsistent_mask][['RID'] + race_cols].head())

# 9. Summary finale
print(f"\n{'=' * 80}")
print("SUMMARY FINALE:")
print(f"{'=' * 80}")
print(f"✓ Colonne RACE rigenerate da df_demog")
print(f"✓ Nessuna riga aggiunta a df_merge")
print(f"✓ RACE uguale per tutte le righe dello stesso RID")
print(f"✓ RID non in df_demog: valori originali mantenuti ({stats['mantenuti_originali']} righe)")


In [ ]:
# ===================================================================
# Step 8: Validation
# ===================================================================
print("\n" + "=" * 80)
print("STEP 8: VALIDATION")
print("=" * 80)

def find_conflicting_subjects(df, dummy_cols):
    df_dummies = df[['RID'] + dummy_cols].copy()
    for col in dummy_cols:
        df_dummies[col] = pd.to_numeric(df_dummies[col], errors='coerce')

    all_nan_mask = df_dummies[dummy_cols].isna().all(axis=1)
    df_dummies = df_dummies[~all_nan_mask]
    df_dummies['sum_dummies'] = df_dummies[dummy_cols].sum(axis=1)

    all_zeros = df_dummies[df_dummies['sum_dummies'] == 0]
    multiple_ones = df_dummies[df_dummies['sum_dummies'] > 1]

    return {
        'all_zeros_count': len(all_zeros['RID'].unique()),
        'multiple_ones_count': len(multiple_ones['RID'].unique())
    }

print()
validation_results = {}
for category, cols in cols_check.items():
    result = find_conflicting_subjects(df_merge, cols)
    validation_results[category] = result
    print(f"{category}:")
    print(f"  - All-zeros: {result['all_zeros_count']} RIDs")
    print(f"  - Multiple-ones: {result['multiple_ones_count']} RIDs")

total_all_zeros = sum(res['all_zeros_count'] for res in validation_results.values())
if total_all_zeros == 0:
    print("\n✓ SUCCESS: No more all-zero rows found!")
else:
    print(f"\n⚠ WARNING: Still {total_all_zeros} all-zero rows remaining")


In [ ]:

# ===================================================================
# Step 9: Data Integrity Check
# ===================================================================
print("\n" + "=" * 80)
print("STEP 9: DATA INTEGRITY CHECK")
print("=" * 80)

print(f"\nChecking {len(all_dummy_cols)} dummy columns...")

issues = []
for col in all_dummy_cols:
    unique_vals = df_merge[col].dropna().unique()
    invalid_vals = [v for v in unique_vals if v not in [0.0, 1.0]]
    if invalid_vals:
        issues.append({'column': col, 'invalid_values': invalid_vals})

if issues:
    print(f"⚠ Found {len(issues)} columns with invalid values:")
    for issue in issues:
        print(f"  - {issue['column']}: {issue['invalid_values']}")
else:
    print("✓ All dummy columns contain only valid values (0.0, 1.0, or NaN)")

print("\nNaN percentages by category:")
for category, cols in cols_check.items():
    nan_pct = df_merge[cols].isna().mean().mean() * 100
    print(f"  {category}: {nan_pct:.2f}%")


In [ ]:
df_merge

In [ ]:
df_merge.columns

In [ ]:
# ===================================================================
# VERIFICA CONSISTENZA ETHNICITY per RID (ignorando NaN)
# ===================================================================

print("=" * 80)
print("VERIFICA CONSISTENZA ETHNICITY PER RID")
print("=" * 80)

ethnicity_cols = cols_check['ETHNICITY']

# Lista per RID inconsistenti
inconsistent_rids = []
rid_details = []

for rid in df_merge['RID'].unique():
    # Estrai tutte le righe di questo RID
    rid_data = df_merge[df_merge['RID'] == rid][['RID', 'VISCODE', 'EXAMDATE'] + ethnicity_cols].copy()
    
    # Filtra solo righe NON tutte NaN per ETHNICITY
    non_nan_mask = ~rid_data[ethnicity_cols].isna().all(axis=1)
    rid_non_nan = rid_data[non_nan_mask]
    
    # Se ha almeno 2 righe con valori non-NaN, verifica consistenza
    if len(rid_non_nan) > 1:
        # Prendi la prima riga non-NaN come riferimento
        first_values = rid_non_nan[ethnicity_cols].iloc[0]
        
        # Confronta con tutte le altre righe non-NaN
        for idx in range(1, len(rid_non_nan)):
            current_values = rid_non_nan[ethnicity_cols].iloc[idx]
            
            # Confronto ignorando NaN
            for col in ethnicity_cols:
                val1 = first_values[col]
                val2 = current_values[col]
                
                # Se entrambi non sono NaN e sono diversi → inconsistenza
                if pd.notna(val1) and pd.notna(val2) and val1 != val2:
                    inconsistent_rids.append(rid)
                    rid_details.append({
                        'RID': rid,
                        'num_righe_totali': len(rid_data),
                        'num_righe_con_valori': len(rid_non_nan),
                        'valori_diversi': rid_non_nan[ethnicity_cols].drop_duplicates().values.tolist()
                    })
                    break
            
            # Se trovata inconsistenza, esci dal loop di questo RID
            if rid in inconsistent_rids:
                break

# ============================================================
# RISULTATI
# ============================================================
print("\n" + "─" * 80)
print("RISULTATI:")
print("─" * 80)

total_rids = df_merge['RID'].nunique()
consistent_rids = total_rids - len(inconsistent_rids)

print(f"RID totali: {total_rids}")
print(f"RID con ETHNICITY consistente: {consistent_rids} ({consistent_rids/total_rids*100:.1f}%)")
print(f"RID con ETHNICITY INCONSISTENTE: {len(inconsistent_rids)} ({len(inconsistent_rids)/total_rids*100:.1f}%)")

if len(inconsistent_rids) == 0:
    print("\n✓ PERFETTO: Tutti i RID hanno ETHNICITY consistente!")
else:
    print(f"\n⚠️ Trovati {len(inconsistent_rids)} RID con valori ETHNICITY contrastanti")
    print(f"\nPrimi 20 RID inconsistenti:")
    print(inconsistent_rids[:20])
    
    # Mostra dettagli dei primi 5 RID problematici
    print(f"\n{'─' * 80}")
    print("DETTAGLIO PRIMI 5 RID INCONSISTENTI:")
    print(f"{'─' * 80}")
    
    for detail in rid_details[:5]:
        print(f"\n  RID {detail['RID']}:")
        print(f"    - Righe totali: {detail['num_righe_totali']}")
        print(f"    - Righe con valori ETHNICITY: {detail['num_righe_con_valori']}")
        print(f"    - Valori diversi trovati: {detail['valori_diversi']}")
        
        # Mostra le righe di questo RID
        rid_sample = df_merge[df_merge['RID'] == detail['RID']][['RID', 'VISCODE', 'EXAMDATE'] + ethnicity_cols]
        print(f"\n    Righe:")
        print(rid_sample.to_string(index=False))

# ============================================================
# RACCOMANDAZIONE
# ============================================================
if len(inconsistent_rids) > 0:
    print(f"\n{'=' * 80}")
    print("RACCOMANDAZIONE:")
    print(f"{'=' * 80}")
    print(f"\n{len(inconsistent_rids)} RID hanno valori ETHNICITY contrastanti tra visite")
    print("\nOpzioni:")
    print("1. Usa df_demog per correggere (se ETHNICITY presente)")
    print("2. Usa il valore più frequente per ogni RID")
    print("3. Imposta tutti a NaN per questi RID (conservativo)")
    print("4. Usa forward/backward fill per propagare il primo valore valido")


In [ ]:
# ===================================================================
# FILL NaN in ETHNICITY usando valori dello stesso RID
# ===================================================================

print("=" * 80)
print("FILL NaN IN ETHNICITY USANDO VALORI DELLO STESSO RID")
print("=" * 80)

ethnicity_cols = cols_check['ETHNICITY']

# ============================================================
# STEP 1: Conta NaN PRIMA
# ============================================================
print("\n" + "─" * 80)
print("SITUAZIONE PRIMA:")
print("─" * 80)

nan_rows_before = df_merge[ethnicity_cols].isna().all(axis=1).sum()
total_rows = len(df_merge)
rids_with_all_nan = df_merge[df_merge[ethnicity_cols].isna().all(axis=1)]['RID'].nunique()

print(f"Righe con ETHNICITY tutto NaN: {nan_rows_before} / {total_rows} ({nan_rows_before/total_rows*100:.1f}%)")
print(f"RID con almeno una riga tutto NaN: {rids_with_all_nan}")

# ============================================================
# STEP 2: Identifica RID con valori inconsistenti
# ============================================================
print("\n" + "─" * 80)
print("VERIFICA CONSISTENZA:")
print("─" * 80)

inconsistent_rids = set()

for rid in df_merge['RID'].unique():
    rid_data = df_merge[df_merge['RID'] == rid][ethnicity_cols]
    
    # Filtra righe non-NaN
    non_nan_rows = rid_data.dropna(how='all')
    
    if len(non_nan_rows) > 1:
        # Verifica se tutte le righe non-NaN sono uguali
        first_row = non_nan_rows.iloc[0]
        for idx in range(1, len(non_nan_rows)):
            if not non_nan_rows.iloc[idx].equals(first_row):
                inconsistent_rids.add(rid)
                break

print(f"RID con valori ETHNICITY inconsistenti: {len(inconsistent_rids)}")
print(f"Questi RID NON verranno riempiti (troppo rischioso)")

# ============================================================
# STEP 3: Fill NaN per RID consistenti
# ============================================================
print("\n" + "─" * 80)
print("RIEMPIMENTO NaN:")
print("─" * 80)

# Ordina per RID e VISCODE/EXAMDATE
sort_cols = ['RID']
if 'VISCODE' in df_merge.columns:
    sort_cols.append('VISCODE')
elif 'EXAMDATE' in df_merge.columns:
    sort_cols.append('EXAMDATE')

# Crea una copia ordinata per il fill
df_sorted = df_merge.sort_values(sort_cols).copy()

# Per ogni colonna ETHNICITY
for col in ethnicity_cols:
    # Forward fill per gruppo RID
    filled_forward = df_sorted.groupby('RID')[col].ffill()
    # Backward fill per gruppo RID
    filled_both = df_sorted.groupby('RID')[col].bfill()
    
    # Applica SOLO se il RID è consistente
    for rid in df_merge['RID'].unique():
        if rid not in inconsistent_rids:
            mask = df_merge['RID'] == rid
            df_merge.loc[mask, col] = filled_both[df_sorted['RID'] == rid].values

print(f"✓ Fill applicato a {df_merge['RID'].nunique() - len(inconsistent_rids)} RID consistenti")
print(f"✗ Fill NON applicato a {len(inconsistent_rids)} RID inconsistenti")

# ============================================================
# STEP 4: Conta NaN DOPO
# ============================================================
print("\n" + "─" * 80)
print("SITUAZIONE DOPO:")
print("─" * 80)

nan_rows_after = df_merge[ethnicity_cols].isna().all(axis=1).sum()
rids_with_all_nan_after = df_merge[df_merge[ethnicity_cols].isna().all(axis=1)]['RID'].nunique()

print(f"Righe con ETHNICITY tutto NaN: {nan_rows_after} / {total_rows} ({nan_rows_after/total_rows*100:.1f}%)")
print(f"RID con almeno una riga tutto NaN: {rids_with_all_nan_after}")

# ============================================================
# STEP 5: Summary
# ============================================================
print("\n" + "=" * 80)
print("SUMMARY:")
print("=" * 80)

filled_rows = nan_rows_before - nan_rows_after
filled_rids = rids_with_all_nan - rids_with_all_nan_after

print(f"Righe riempite: {filled_rows} / {nan_rows_before}")
print(f"RID completamente riempiti: {filled_rids}")

if filled_rows > 0:
    print(f"✓ Riempimento riuscito per {filled_rows} righe ({filled_rows/nan_rows_before*100:.1f}%)")
else:
    print("⚠️ Nessuna riga riempita")

if nan_rows_after > 0:
    print(f"\nRimangono {nan_rows_after} righe con NaN:")
    print(f"  - {len(inconsistent_rids)} RID inconsistenti (non riempiti)")
    print(f"  - Altri RID senza alcun valore ETHNICITY disponibile")


In [ ]:

# ===================================================================
# Step 10: Save Corrected Dataset
# ===================================================================
print("\n" + "=" * 80)
print("STEP 10: SAVING CORRECTED DATASET")
print("=" * 80)

print("\nSaving to datalake...")

original_metadata = client.get_metadata('cleaned/merged/combination/subMERGE_4-3_elecsys_FBP.csv')
metadata = original_metadata['metadata']['custom'].copy()
metadata['dummy_fix_date'] = pd.Timestamp.now().isoformat()

result = client.upload_dataframe(
    df=df_merge,
    object_name='subMERGE_4-3_elecsys_FBP_fxd_0.csv',
    prefix='cleaned/merged/combination',
    metadata=metadata
)

print("✓ Upload complete!")
print(f"\nDataset details:")
print(f"  - Rows: {len(df_merge)}")
print(f"  - Columns: {len(df_merge.columns)}")
print(f"  - File: cleaned/merged/combination/subMERGE_4-3_elecsys_FBP.csv")

print("\n" + "=" * 80)
print("ALL STEPS COMPLETED SUCCESSFULLY!")
print("=" * 80)


